In [2]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


X = pd.read_csv("../data/secom.data", sep=" ", header=None)
y = pd.read_csv("../data/secom_labels.data", sep =" ", header=None,
                names =["label", "timestamp"])
y["timestamp"] = pd.to_datetime(y["timestamp"], format="%d/%m/%Y %H:%M:%S")

# the label 0/1 ( 1 = fail) and sort by time
df = X.copy()
df["label"]= (y["label"] == 1).astype(int)
df["timestamp"] = y["timestamp"]
df = df.sort_values("timestamp").reset_index(drop=True)

# time  based split
# 80% for training 
# 20% for testing
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]

feature_cols = [c for c in df.columns if c not in ("label", "timestamp")]
X_train, y_train = train[feature_cols], train["label"]
X_test, y_test = test[feature_cols], test["label"]

# drop useless columns
keep = [c for c in feature_cols
        if X_train[c].isna().mean() <= 0.5 and X_train[c].nunique() > 1]
X_train, X_test = X_train[keep], X_test[keep]
print(f"features: {len(feature_cols)} -> {len(keep)}")


# fill missing values with the training median
# scale
imputer =  SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train_p = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_p = scaler.transform(imputer.transform(X_test))

print("fail rate  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))



features: 590 -> 444
fail rate  train: 0.069  test: 0.054


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (recall_score, precision_score, f1_score,
                             matthews_corrcoef, confusion_matrix)
from imblearn.over_sampling import SMOTE

def report(name, model, X_te, y_te):
    pred = model.predict(X_te)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    print(f"{name:28s} recall ={recall_score(y_te, pred): .3f}"
          f"precision={precision_score(y_te, pred, zero_division=0):.3f} "
          f"F1={f1_score(y_te, pred):.3f} MCC={matthews_corrcoef(y_te, pred):.3f} "
          f"(caught {tp}/{tp+fn}, false alarms {fp})")


# baseline reference
base = LogisticRegression(max_iter=1000).fit(X_train_p, y_train)
report("baseline", base, X_test_p, y_test)

# A. class weights, with a few regularization strengths (smaller C = simpler  model)
for C in [1, 0.1, 0.01]:
    m = LogisticRegression(class_weight="balanced", C=C, max_iter = 2000)
    m.fit(X_train_p, y_train)
    report(f"class_weight, C={C}", m, X_test_p, y_test)

# B. smote 
X_sm, y_sm = SMOTE(random_state=42).fit_resample(X_train_p, y_train)
for C in [1,0.1,0.01]:
    m = LogisticRegression(C=C, max_iter=2000).fit(X_sm, y_sm)
    report(f"SMOTE, C={C}", m, X_test_p, y_test)

baseline                     recall = 0.176precision=0.300 F1=0.222 MCC=0.197 (caught 3/17, false alarms 7)
class_weight, C=1            recall = 0.176precision=0.188 F1=0.182 MCC=0.137 (caught 3/17, false alarms 13)
class_weight, C=0.1          recall = 0.235precision=0.211 F1=0.222 MCC=0.175 (caught 4/17, false alarms 15)
class_weight, C=0.01         recall = 0.235precision=0.211 F1=0.222 MCC=0.175 (caught 4/17, false alarms 15)
SMOTE, C=1                   recall = 0.235precision=0.267 F1=0.250 MCC=0.210 (caught 4/17, false alarms 11)
SMOTE, C=0.1                 recall = 0.235precision=0.222 F1=0.229 MCC=0.183 (caught 4/17, false alarms 14)
SMOTE, C=0.01                recall = 0.235precision=0.235 F1=0.235 MCC=0.192 (caught 4/17, false alarms 13)
